# Low-light × weather S1 replication (protocol v1.1)

Import this notebook from GitHub, enable Internet and a GPU, then choose **Save Version → Save & Run All**. No dataset upload or training is needed. The notebook downloads the pinned public CDD-11 test archive and frozen OneRestore/MIRAGE checkpoints, verifies their hashes, runs CPU checks and a 3-scene discovery smoke, and evaluates the 39 confirmation scenes only if the measured estimate is within the 4 GPU-hour cap. It never opens the 78-scene holdout. Download the printed `low_weather_replication_results_<run-id>.zip`, including when a stage fails. The GT-fitted affine output is diagnostic only and is never counted as deployed restoration quality.


In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import shutil
import subprocess
import sys
import traceback
import zipfile
from pathlib import Path

PROJECT_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
PROJECT_COMMIT = "50aeed6"
WORK = Path("/kaggle/working/low_weather_replication")
PROJECT = WORK / "project"
DATA_WORK = WORK / "data_work"
MAX_GPU_HOURS = 4.0
MAX_SMOKE_SCENES = 3
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BUNDLE = WORK / "bundles" / RUN_ID
LOGS = BUNDLE / "logs"
ARCHIVE = Path("/kaggle/working") / f"low_weather_replication_results_{RUN_ID}.zip"
BUNDLE.mkdir(parents=True, exist_ok=False)
LOGS.mkdir(parents=True, exist_ok=True)
errors = []
stage = "start"
summary_result = None
budget_result = None
confirmation_started = False
project_revision = None

def run_logged(args, name, cwd=None):
    log_path = LOGS / f"{name}.log"
    command = [str(part) for part in args]
    print(f"[{name}] {' '.join(command)}", flush=True)
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"{name} failed with exit code {return_code}; see {log_path}")
    return log_path

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def copy_if_exists(source, destination):
    source = Path(source)
    if source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    elif source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)

def assemble_bundle(final_status):
    if PROJECT.is_dir():
        for relative in (
            "configs/low_weather_replication_v1.json",
            "configs/low_weather_replication/manifest_fixture.json",
            "configs/low_weather_replication/source_fixture.json",
        ):
            copy_if_exists(PROJECT / relative, BUNDLE / Path(relative).name)
    for relative in ("manifest.json", "sources.json"):
        copy_if_exists(DATA_WORK / relative, BUNDLE / relative)
    copy_if_exists(DATA_WORK / "s1", BUNDLE / "s1")
    if PROJECT.is_dir() and (PROJECT / "configs/low_weather_replication/manifest_fixture.json").is_file():
        config_path = PROJECT / "configs/low_weather_replication_v1.json"
        fixture_path = PROJECT / "configs/low_weather_replication/manifest_fixture.json"
        source_fixture_path = PROJECT / "configs/low_weather_replication/source_fixture.json"
        protocol = json.loads(config_path.read_text(encoding="utf-8"))
        fixture_data = json.loads(fixture_path.read_text(encoding="utf-8"))
        protocol["manifest_fixture_sha256"] = fixture_data["source_manifest_sha256"]
        protocol["manifest_fixture_file_sha256"] = sha256_file(fixture_path)
        protocol["source_fixture_sha256"] = sha256_file(source_fixture_path)
        protocol["project_code_commit"] = project_revision
        (BUNDLE / "protocol.json").write_text(
            json.dumps(protocol, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
        )
    try:
        freeze = subprocess.run(
            [sys.executable, "-m", "pip", "freeze"],
            check=False, capture_output=True, text=True, timeout=120,
        )
        environment = freeze.stdout
        if freeze.returncode:
            environment += f"\npip freeze exit code: {freeze.returncode}\n{freeze.stderr}"
    except Exception as exc:
        environment = f"Could not capture pip freeze: {type(exc).__name__}: {exc}"
    (BUNDLE / "environment.txt").write_text(environment, encoding="utf-8")
    run_data = {
        "run_id_utc": RUN_ID,
        "status": final_status,
        "stage": stage,
        "errors": list(errors),
        "project_url": PROJECT_URL,
        "project_code_commit": project_revision,
        "gpu": gpu_name if "gpu_name" in globals() else None,
        "max_gpu_hours": MAX_GPU_HOURS,
        "budget_estimate": budget_result,
        "confirmation_started": confirmation_started,
        "confirmation_completed": all(
            (DATA_WORK / "s1/confirmation" / model / "status.json").is_file()
            and json.loads((DATA_WORK / "s1/confirmation" / model / "status.json").read_text()).get("status") == "complete"
            for model in ("onerestore", "mirage")
        ),
        "summary_status": summary_result.get("status") if isinstance(summary_result, dict) else None,
        "holdout_content_opened": False,
        "archive_only_contains_results_and_metadata": True,
    }
    (BUNDLE / "run.json").write_text(
        json.dumps(run_data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
    )
    if ARCHIVE.exists():
        ARCHIVE.unlink()
    shutil.make_archive(str(ARCHIVE.with_suffix("")), "zip", root_dir=BUNDLE)

try:
    stage = "clone_pinned_project"
    WORK.mkdir(parents=True, exist_ok=True)
    if PROJECT.exists():
        if not (PROJECT / ".git").exists():
            raise RuntimeError(f"Project path exists but is not a Git checkout: {PROJECT}")
        existing = subprocess.check_output(
            ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
        ).strip()
        if existing != PROJECT_COMMIT:
            raise RuntimeError(
                f"Existing project checkout is {existing}, expected pinned {PROJECT_COMMIT}; "
                "preserving it and stopping instead of replacing local Kaggle files."
            )
        dirty = subprocess.check_output(
            ["git", "-C", str(PROJECT), "status", "--porcelain"], text=True
        ).strip()
        if dirty:
            raise RuntimeError("Pinned project checkout has local edits; preserving it and stopping.")
    else:
        run_logged(["git", "clone", PROJECT_URL, PROJECT], "clone_project")
        run_logged(["git", "-C", PROJECT, "checkout", "--detach", PROJECT_COMMIT], "checkout_project")
    project_revision = subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip()
    if not project_revision.startswith(PROJECT_COMMIT):
        raise RuntimeError("Could not verify the pinned S1 code commit.")
    os.chdir(PROJECT)

    stage = "install_dependencies"
    run_logged([
        sys.executable, "-m", "pip", "install",
        "huggingface_hub", "gdown", "einops", "timm", "fvcore", "thop",
        "scikit-image", "opencv-python-headless",
    ], "install_dependencies", cwd=PROJECT)

    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU before running this notebook.")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Code commit: {project_revision}; GPU: {gpu_name}", flush=True)

    stage = "cpu_unit_tests"
    run_logged([
        sys.executable, "-m", "unittest",
        "hybrid_cot_nafnet.test_low_weather_generator_b",
        "hybrid_cot_nafnet.test_low_weather_replication",
        "hybrid_cot_nafnet.test_low_weather_summary",
        "hybrid_cot_nafnet.test_low_weather_interaction",
        "-v",
    ], "cpu_unit_tests", cwd=PROJECT)

    stage = "prepare_pinned_sources_and_manifest"
    run_logged([
        sys.executable, "-u", "-m", "hybrid_cot_nafnet.audit_low_weather_replication",
        "prepare", "--work", DATA_WORK, "--project", PROJECT,
        "--fixture", PROJECT / "configs/low_weather_replication/source_fixture.json",
        "--manifest-fixture", PROJECT / "configs/low_weather_replication/manifest_fixture.json",
    ], "prepare", cwd=PROJECT)

    stage = "input_only_preflight"
    run_logged([
        sys.executable, "-u", "-m", "hybrid_cot_nafnet.audit_low_weather_replication",
        "preflight", "--work", DATA_WORK, "--max-scenes", MAX_SMOKE_SCENES,
        "--manifest-fixture", PROJECT / "configs/low_weather_replication/manifest_fixture.json",
    ], "preflight", cwd=PROJECT)

    stage = "discovery_gpu_smoke"
    for model_name in ("onerestore", "mirage"):
        run_logged([
            sys.executable, "-u", "-m", "hybrid_cot_nafnet.audit_low_weather_replication",
            "evaluate", "--work", DATA_WORK, "--model", model_name,
            "--partition", "discovery", "--max-scenes", MAX_SMOKE_SCENES,
            "--manifest-fixture", PROJECT / "configs/low_weather_replication/manifest_fixture.json",
        ], f"smoke_{model_name}", cwd=PROJECT)

    from hybrid_cot_nafnet.audit_low_weather_replication import estimate_confirmation_gpu_hours
    smoke_statuses = {
        model_name: json.loads(
            (DATA_WORK / "s1/discovery" / model_name / "status.json").read_text(encoding="utf-8")
        )
        for model_name in ("onerestore", "mirage")
    }
    budget_result = estimate_confirmation_gpu_hours(smoke_statuses)
    config = json.loads((PROJECT / "configs/low_weather_replication_v1.json").read_text(encoding="utf-8"))
    if float(config["runtime_budget_gpu_hours"]) != MAX_GPU_HOURS:
        raise RuntimeError("Notebook GPU-hour cap differs from the locked S1 config.")
    if budget_result["estimate_gpu_hours"] > MAX_GPU_HOURS:
        stage = "stopped_runtime_budget"
        print(
            f"Confirmation skipped: conservative estimate {budget_result['estimate_gpu_hours']:.2f} "
            f"GPU-hours exceeds the {MAX_GPU_HOURS:.1f}-hour cap.",
            flush=True,
        )
    else:
        stage = "confirmation_gpu_evaluation"
        confirmation_started = True
        for model_name in ("onerestore", "mirage"):
            run_logged([
                sys.executable, "-u", "-m", "hybrid_cot_nafnet.audit_low_weather_replication",
                "evaluate", "--work", DATA_WORK, "--model", model_name,
                "--partition", "confirmation",
                "--manifest-fixture", PROJECT / "configs/low_weather_replication/manifest_fixture.json",
            ], f"confirmation_{model_name}", cwd=PROJECT)

    completed = confirmation_started and all(
        (DATA_WORK / "s1/confirmation" / model / "status.json").is_file()
        and json.loads((DATA_WORK / "s1/confirmation" / model / "status.json").read_text()).get("status") == "complete"
        for model in ("onerestore", "mirage")
    )
    final_status = "CONFIRMATION_COMPLETE" if completed else (
        "STOPPED_RUNTIME_BUDGET" if stage == "stopped_runtime_budget" else "PARTIAL"
    )
except Exception:
    errors.append(traceback.format_exc())
    final_status = "FAILED_OR_PARTIAL"
finally:
    try:
        assemble_bundle(final_status)
    except Exception:
        errors.append("Artifact assembly failed:\n" + traceback.format_exc())

if not errors and confirmation_started:
    try:
        stage = "cpu_summary"
        run_logged([
            sys.executable, "-u", "-m", "hybrid_cot_nafnet.summarize_low_weather_replication",
            "--zip", ARCHIVE,
            "--output", BUNDLE / "s1_summary.json",
            "--report", BUNDLE / "s1_summary.md",
        ], "summary", cwd=PROJECT)
        summary_result = json.loads((BUNDLE / "s1_summary.json").read_text(encoding="utf-8"))
        final_status = summary_result["status"]
    except Exception:
        errors.append(traceback.format_exc())
        final_status = "SUMMARY_FAILED"
    try:
        assemble_bundle(final_status)
    except Exception:
        errors.append("Final artifact assembly failed:\n" + traceback.format_exc())

print(f"Download artifact: {ARCHIVE}", flush=True)
print(f"Run status: {final_status}; errors={len(errors)}", flush=True)
if errors:
    raise RuntimeError(f"S1 run had errors; download the ZIP first: {ARCHIVE}")
if final_status == "STOPPED_RUNTIME_BUDGET":
    print("The notebook stopped before confirmation. Send the ZIP for review.", flush=True)
elif summary_result is not None:
    print(f"S1 confirmation screen: {summary_result['status']}", flush=True)
